Cell 1 — Reload + Better Label Creation

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_parquet('../data/combined_dataset.parquet')
print(f"✅ Loaded: {df.shape[0]:,} rows | {df['PatientID'].nunique():,} patients")

# Better feature list — include more available columns
FEATURES = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp',
            'WBC', 'Creatinine', 'Bilirubin_total', 'Lactate',
            'Glucose', 'Hgb', 'pH', 'Age', 'HospAdmTime', 'ICULOS']

available = [f for f in FEATURES if f in df.columns]
print(f"✅ Features available: {len(available)} — {available}")

cols = ['PatientID', 'Hour', 'SepsisLabel'] + available
data = df[cols].copy()

# Impute missing — forward fill then median
data[available] = data.groupby('PatientID')[available].transform(
    lambda x: x.ffill().bfill()
)
for col in available:
    data[col].fillna(data[col].median(), inplace=True)

print(f"✅ Missing values handled")
print(f"   Remaining nulls: {data[available].isnull().sum().sum()}")

✅ Loaded: 1,552,210 rows | 40,336 patients
✅ Features available: 17 — ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'WBC', 'Creatinine', 'Bilirubin_total', 'Lactate', 'Glucose', 'Hgb', 'pH', 'Age', 'HospAdmTime', 'ICULOS']
✅ Missing values handled
   Remaining nulls: 0


Cell 2 — Correct Label + Split

In [3]:
from sklearn.model_selection import GroupShuffleSplit

# CORRECT approach: use SepsisLabel directly as target
# The label already means "patient has/will have sepsis"
# We predict at the ROW level — each hour is one prediction
X = data[available].values
y = data['SepsisLabel'].values
groups = data['PatientID'].values

print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")
print(f"Positive rate: {y.mean()*100:.1f}%")

# Split by PATIENT — not by row — to prevent data leakage
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"\nTrain: {X_train.shape[0]:,} rows | {y_train.sum():,} positive")
print(f"Test : {X_test.shape[0]:,} rows | {y_test.sum():,} positive")

# SMOTE on train only
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print(f"\n✅ After SMOTE — Train: {X_train_sm.shape[0]:,} rows | Balanced: {y_train_sm.mean()*100:.0f}%")

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_sm)
X_test_sc  = scaler.transform(X_test)

X shape : (1552210, 17)
y shape : (1552210,)
Positive rate: 1.8%

Train: 1,241,213 rows | 22,669 positive
Test : 310,997 rows | 5,247 positive

✅ After SMOTE — Train: 2,437,088 rows | Balanced: 50%


Cell 3 — Retrain + Better Results

In [5]:
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report
import joblib

# Logistic Regression
print("Training Logistic Regression...")
lr = LogisticRegression(max_iter=2000, C=0.1, random_state=42, n_jobs=-1)
lr.fit(X_train_sc, y_train_sm)
lr_proba = lr.predict_proba(X_test_sc)[:, 1]
lr_auroc = roc_auc_score(y_test, lr_proba)

# XGBoost — FIXED
# Remove scale_pos_weight since SMOTE already balanced the data
# Use early stopping to prevent overfitting
# Evaluate on X_test_sc (scaled), not X_test
print("Training XGBoost...")
xgb = XGBClassifier(
    n_estimators=1000,       # high ceiling — early stopping will cut it
    max_depth=6,             # reduced from 7 to reduce overfitting
    learning_rate=0.02,      # slower learning = better generalization
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=10,     # increased — more conservative splits
    scale_pos_weight=1,      # REMOVED — SMOTE already balanced, don't double
    eval_metric='auc',
    early_stopping_rounds=50, # stop if no improvement for 50 rounds
    random_state=42,
    n_jobs=-1
)

xgb.fit(
    X_train_sc, y_train_sm,
    eval_set=[(X_test_sc, y_test)],  # FIXED — use scaled test set
    verbose=100
)

xgb_proba = xgb.predict_proba(X_test_sc)[:, 1]   # FIXED — scaled
xgb_auroc = roc_auc_score(y_test, xgb_proba)
xgb_pred  = (xgb_proba >= 0.4).astype(int)        # raised threshold
lr_pred   = (lr_proba >= 0.4).astype(int)

print(f"\n{'='*45}")
print(f"FIXED RESULTS")
print(f"{'='*45}")
print(f"Logistic Regression AUROC : {lr_auroc:.4f}")
print(f"XGBoost AUROC             : {xgb_auroc:.4f}")
print(f"Best XGBoost iteration    : {xgb.best_iteration}")
print(f"{'='*45}")
print("\nXGBoost Classification Report:")
print(classification_report(y_test, xgb_pred,
      target_names=['No Sepsis', 'Sepsis']))

# Save
joblib.dump(xgb,    '../models/xgboost_model.pkl')
joblib.dump(lr,     '../models/logistic_regression.pkl')
joblib.dump(scaler, '../models/scaler.pkl')

import json
with open('../models/feature_list.json', 'w') as f:
    json.dump(available, f)

print("\n✅ Models + feature list saved")
print(f"✅ XGBoost stopped at tree #{xgb.best_iteration}")

Training Logistic Regression...
Training XGBoost...
[0]	validation_0-auc:0.69012
[100]	validation_0-auc:0.81370
[198]	validation_0-auc:0.81500

FIXED RESULTS
Logistic Regression AUROC : 0.7336
XGBoost AUROC             : 0.8158
Best XGBoost iteration    : 148

XGBoost Classification Report:
              precision    recall  f1-score   support

   No Sepsis       0.99      0.74      0.85    305750
      Sepsis       0.05      0.73      0.09      5247

    accuracy                           0.74    310997
   macro avg       0.52      0.74      0.47    310997
weighted avg       0.98      0.74      0.84    310997


✅ Models + feature list saved
✅ XGBoost stopped at tree #148


CELL 4 — Save test data for SHAP analysis

In [ ]:
# Save test data as .npy files for downstream analysis (SHAP, etc)
np.save('../data/X_test.npy', X_test_sc)          # scaled features
np.save('../data/y_test.npy', y_test)             # true labels
np.save('../data/xgb_proba_test.npy', xgb_proba)  # model predictions

print("✅ Saved test data:")
print(f"   ../data/X_test.npy         — shape {X_test_sc.shape}")
print(f"   ../data/y_test.npy         — shape {y_test.shape}")
print(f"   ../data/xgb_proba_test.npy — shape {xgb_proba.shape}")